In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
ratings_path = "ratings.csv"   # 경로만 네 환경에 맞게 수정
movies_path  = "movies.csv"    # 없으면 추천 타이틀 부분만 빼고 써도 됨

ratings_df = pd.read_csv(ratings_path)
movies_df  = pd.read_csv(movies_path)
print(ratings_df.head())
print(movies_df.head())


   userId  movieId  rating
0       1        7       3
1       1       16       1
2       1       11       5
3       1       24       3
4       1       21       3
   movieId    title
0        1  Movie 1
1        2  Movie 2
2        3  Movie 3
3        4  Movie 4
4        5  Movie 5


In [4]:
def encode_ids(ratings_df):
    # 유저 / 아이템 유니크 값
    unique_users = ratings_df["userId"].unique()
    unique_items = ratings_df["movieId"].unique()

    user2idx = {u: i for i, u in enumerate(unique_users)}
    idx2user = {i: u for u, i in user2idx.items()}

    item2idx = {m: i for i, m in enumerate(unique_items)}
    idx2item = {i: m for m, i in item2idx.items()}

    # 새로운 컬럼으로 인덱스 저장
    ratings_df = ratings_df.copy()
    ratings_df["user_idx"] = ratings_df["userId"].map(user2idx)
    ratings_df["item_idx"] = ratings_df["movieId"].map(item2idx)

    return ratings_df, user2idx, idx2user, item2idx, idx2item

ratings_df, user2idx, idx2user, item2idx, idx2item = encode_ids(ratings_df)

num_users = len(user2idx)
num_items = len(item2idx)
print("num_users:", num_users, " num_items:", num_items)


num_users: 40  num_items: 30


In [5]:
class MovieRatingDataset(Dataset):
    def __init__(self, df):
        self.user_idx = df["user_idx"].values.astype(np.int64)
        self.item_idx = df["item_idx"].values.astype(np.int64)
        self.ratings  = df["rating"].values.astype(np.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.user_idx[idx],
            self.item_idx[idx],
            self.ratings[idx],
        )

# 간단하게 80:20 train/val 분리
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(ratings_df, test_size=0.2, random_state=42)

train_dataset = MovieRatingDataset(train_df)
val_dataset   = MovieRatingDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1024, shuffle=False)


In [6]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=50):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        # 초기화 (선택)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx):
        u = self.user_emb(user_idx)
        i = self.item_emb(item_idx)
        u_b = self.user_bias(user_idx).squeeze(-1)
        i_b = self.item_bias(item_idx).squeeze(-1)

        dot = (u * i).sum(dim=1)
        out = dot + u_b + i_b

        # 평점 스케일(예: 0.5~5점)로 맞춰 주고 싶으면 clamp
        # out = torch.clamp(out, 0.5, 5.0)
        return out


In [7]:
model = MatrixFactorization(num_users, num_items, embedding_dim=50).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for user_idx, item_idx, rating in loader:
        user_idx = user_idx.to(device)
        item_idx = item_idx.to(device)
        rating   = rating.to(device)

        optimizer.zero_grad()
        preds = model(user_idx, item_idx)
        loss = criterion(preds, rating)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * rating.size(0)

    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.inference_mode():
        for user_idx, item_idx, rating in loader:
            user_idx = user_idx.to(device)
            item_idx = item_idx.to(device)
            rating   = rating.to(device)

            preds = model(user_idx, item_idx)
            loss = criterion(preds, rating)
            total_loss += loss.item() * rating.size(0)

    return total_loss / len(loader.dataset)

n_epochs = 10  # 처음엔 3~5로 테스트해 보고 늘려 가도 됨

for epoch in range(1, n_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss   = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch:02d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")


Epoch 01 | Train MSE: 11.5646 | Val MSE: 11.6931
Epoch 02 | Train MSE: 11.5506 | Val MSE: 11.6807
Epoch 03 | Train MSE: 11.5367 | Val MSE: 11.6682
Epoch 04 | Train MSE: 11.5226 | Val MSE: 11.6557
Epoch 05 | Train MSE: 11.5085 | Val MSE: 11.6430
Epoch 06 | Train MSE: 11.4941 | Val MSE: 11.6303
Epoch 07 | Train MSE: 11.4796 | Val MSE: 11.6174
Epoch 08 | Train MSE: 11.4648 | Val MSE: 11.6043
Epoch 09 | Train MSE: 11.4497 | Val MSE: 11.5911
Epoch 10 | Train MSE: 11.4344 | Val MSE: 11.5776


In [8]:
# 유저가 이미 본 아이템 set 미리 구해두기
user_seen_items = (
    ratings_df.groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)


In [9]:
def recommend_for_user(
    model,
    raw_user_id,
    user2idx,
    idx2item,
    item2idx,
    ratings_df,
    movies_df,
    top_k=10,
    device=device,
):
    if raw_user_id not in user2idx:
        print("Unknown user")
        return []

    model.eval()
    user_idx = user2idx[raw_user_id]

    # 전체 아이템 인덱스 텐서
    all_item_indices = np.array(list(idx2item.keys()))  # 0 ~ num_items-1
    all_item_indices_t = torch.tensor(all_item_indices, dtype=torch.long, device=device)

    user_tensor = torch.full(
        (len(all_item_indices),),
        fill_value=user_idx,
        dtype=torch.long,
        device=device,
    )

    with torch.inference_mode():
        preds = model(user_tensor, all_item_indices_t)
        preds = preds.cpu().numpy()

    # 이미 본 영화 제외
    seen = user_seen_items.get(raw_user_id, set())
    candidate = []
    for idx, score in zip(all_item_indices, preds):
        movie_id = idx2item[idx]
        if movie_id in seen:
            continue
        candidate.append((movie_id, score))

    # 점수 기준 상위 top_k
    candidate.sort(key=lambda x: x[1], reverse=True)
    top = candidate[:top_k]

    # 타이틀 붙이기
    movie_map = movies_df.set_index("movieId")["title"].to_dict()
    result = []
    for movie_id, score in top:
        title = movie_map.get(movie_id, f"movieId={movie_id}")
        result.append(
            {
                "movieId": movie_id,
                "title": title,
                "pred_score": float(score),
            }
        )
    return result


In [10]:
# 예: userId = 1인 사람에게 10개 추천
user_id_example = 1
recs = recommend_for_user(
    model,
    raw_user_id=user_id_example,
    user2idx=user2idx,
    idx2item=idx2item,
    item2idx=item2idx,
    ratings_df=ratings_df,
    movies_df=movies_df,
    top_k=10,
)

for r in recs:
    print(f"{r['title']} (movieId={r['movieId']}) | score={r['pred_score']:.3f}")


Movie 19 (movieId=19) | score=0.023
Movie 23 (movieId=23) | score=0.023
Movie 15 (movieId=15) | score=0.023
Movie 6 (movieId=6) | score=0.023
Movie 4 (movieId=4) | score=0.022
Movie 5 (movieId=5) | score=0.022
Movie 13 (movieId=13) | score=0.022
Movie 9 (movieId=9) | score=0.022
Movie 20 (movieId=20) | score=0.021
Movie 28 (movieId=28) | score=0.021
